# TP 2 - Préparation de la base RAG


Ce notebook prépare une base documentaire simple pour le RAG, sans optimisation.
On construit ici une version de base (V1) comme point de référence.

### 0.1. Objectif
- **TP 2_1** : Créer une base de données vectorielle (`chroma_db_rag_v1`) à partir de documents Markdown issus de guides de voyage
- **TP 2_2** : Créer un assistant de voyage RAG simple
- **TP 2_3** : Créer une base de données mieux structurée / optimisée (`chroma_db_rag_v2`)
- **TP 2_4** : Créer un assistant de voyage RAG avec des méthodes avancées de retrieval

### 0.2. Documentation générale

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[ChromaDB](https://docs.trychroma.com/)

In [ ]:
from shared.config import ROOT_DIR
from shared.rag_utils import (
    rag_load_markdown_documents,
    rag_chunk_document_by_chars,
    rag_describe_chunks,
    rag_index_chunks_chroma,
    # INFO : Choix entre local ou cloud (embeddings)
    rag_embed_all_chunks, # Cloud
    #rag_embed_all_chunks_local as rag_embed_all_chunks, # Local
)

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
MARKDOWN_DIR = DATA_DIR / "guides_markdown"
CHROMA_DIR_V1 = DATA_DIR / "chroma_db_rag_v1"

### 0.3. Récapitulatif des fonctions utilisées dans ce notebook

**Fournies**

- `rag_load_markdown_documents` : charge les fichiers `.md` d'un dossier en `MarkdownDocument`
- `rag_chunk_document_by_chars` : découpe un document en chunks de taille fixe avec overlap
- `rag_describe_chunks` : affiche des statistiques et la distribution des tailles de chunks
- `rag_embed_all_chunks` : calcule les embeddings de tous les chunks, en appelant `rag_embed_text_batch` par groupes
- `rag_index_chunks_chroma` : indexe des chunks vectorisés dans Chroma

--> Disponibles dans `shared/rag_utils.py`

### 0.4. Charger les documents

Utilisez la fonction `rag_load_markdown_documents` importée depuis `shared/rag_utils.py`.

Elle lit les fichiers `.md` d'un dossier et retourne une liste de `MarkdownDocument` (défini aussi dans le même fichier).

In [ ]:
documents = rag_load_markdown_documents(MARKDOWN_DIR)

total_documents = len(documents)
total_characters = sum(len(document["text"]) for document in documents)

print(f"Documents Markdown chargés : {total_documents}")
print(f"Nombre total de caractères : {total_characters}")

---
## 1. Stratégie de chunking classique (V1)

Un document complet est **trop long** pour la recherche par vecteurs latents.
On le **découpe en chunks** de taille comparable, mesurés en nombre de caractères.
L'**overlap** garde une zone de texte partagée entre deux chunks voisins pour conserver la continuité.

Repère attendu : entre 100 et 300 chunks au total.

### 1.1. Appliquer le chunking à tous les documents

In [ ]:
chunks_v1 = []
for doc in documents:
    chunks_v1.extend(rag_chunk_document_by_chars(doc, chunk_chars=3000, chunk_overlap_chars=250))

### 1.2. Inspecter les chunks (statistiques)

`rag_describe_chunks` affiche des statistiques (nombre de chunks, taille min/max/moyenne) et un histogramme de la distribution des tailles par document source.

In [ ]:
rag_describe_chunks(chunks_v1)

### 1.3. Afficher quelques chunks

Afficher quelques chunks pour comprendre le résultat du découpage.

In [ ]:
sample_indices = [0, len(chunks_v1) // 4, len(chunks_v1) // 2, 3 * len(chunks_v1) // 4, len(chunks_v1) - 1]

for idx in sample_indices:
    chunk = chunks_v1[idx]
    print(f"-- Chunk #{chunk.chunk_id} | source: {chunk.source} | {len(chunk.text)} caractères ---")
    print(chunk.text[:500])
    print("..." if len(chunk.text) > 500 else "")
    print("\n\n\n")

---
## 2. Embeddings et indexation

Dans cette partie, nous allons calculer les embeddings et les indexer dans une base de données Chroma.

**Étape 1 : Calcul des embeddings** - Gemini API

- `rag_embed_text_batch` calcule les embeddings d'un lot (*batch*) de textes (l'API a une limite de textes à envoyer en un seul appel)
- `rag_embed_all_chunks` appelle `rag_embed_text_batch` autant de fois que nécessaire pour calculer les embeddings de tous les chunks (par batch)

**Étape 2 : Indexation des embeddings** - Chroma

- `rag_index_chunks_chroma` indexe les embeddings dans une base de données Chroma (SQLite)

### 2.1. Calculer les embeddings des chunks

In [ ]:
chunk_embeddings_v1 = rag_embed_all_chunks(
    chunks=chunks_v1,
    batch_size=16
)

### 2.2. Indexer les chunks dans Chroma

In [ ]:
rag_index_chunks_chroma(persist_dir=CHROMA_DIR_V1, chunks=chunk_embeddings_v1)

print(f"V1 - Chunks indexés : {len(chunk_embeddings_v1)}")
print(f"V1 - Dimension des embeddings : {len(chunk_embeddings_v1[0].embedding)}")